# 03 - Huan luyen 4 model hoi quy

Huan luyen 4 model cho cung mot bai toan hoi quy (du doan `price`), cung mot cach chia du lieu (`train_test_split(test_size=0.2, random_state=42)`) va cung bo metric (MAE, RMSE, R2) de so sanh cong bang.

| Model | Vai tro |
|---|---|
| Linear Regression | Baseline - de biet 'tot hon ngau nhien/don gian' la the nao |
| Decision Tree | Model phi tuyen don gian, de giai thich |
| Random Forest | Ensemble (bagging) - giam phuong sai so voi 1 cay don |
| SVR (kernel RBF) | Model bien/kernel - nam bat quan he phi tuyen phuc tap |


## Xu ly rieng cho bien target: log1p

`price` lech phai manh (Hinh 1, notebook EDA) nen dung `sklearn.compose.TransformedTargetRegressor` voi `func=np.log1p`, `inverse_func=np.expm1`: model duoc fit tren `log1p(price)`, khi `predict()` ket qua tu dong duoc doi nguoc ve trieu VND bang `expm1`. Cac metric (MAE/RMSE/R2) tinh tren gia tri **da doi nguoc** (don vi that: trieu VND), de con so co y nghia thuc te va so sanh duoc giua cac model.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from data_utils import (load_raw, clean_dataframe, FEATURE_COLUMNS, TARGET_COLUMN,
                         NUMERIC_FEATURES, CATEGORICAL_FEATURES)

RANDOM_STATE = 42
df = clean_dataframe(load_raw('../data/data.csv'))
X, y = df[FEATURE_COLUMNS], df[TARGET_COLUMN].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (19778, 10) Test: (4945, 10)


## Sieu tham so da thu (tom tat)

Da thu nhieu gia tri bang `GridSearchCV`/kiem tra thu cong tren tap train (5-fold), chon cau hinh co MAE/R2 tren validation tot nhat va thoi gian huan luyen/du doan chap nhan duoc de deploy:

| Model | Sieu tham so da thu | Gia tri chon | Ly do |
|---|---|---|
| Decision Tree | `max_depth` in [6,10,12,16,None], `min_samples_leaf` in [1,5,10] | `max_depth=12, min_samples_leaf=5` | Sau 12 do sau, R2 tren validation gan nhu khong tang nhung R2 train tang manh -> dau hieu overfit |
| Random Forest | `n_estimators` in [100,200,300], `max_depth` in [10,16,None] | `n_estimators=300, max_depth=16, min_samples_leaf=2` | 300 cay on dinh hon 100/200 (do lech chuan giua cac fold thap hon), tang tiep khong dang ke ve MAE nhung ton thoi gian/dung luong |
| SVR | `kernel` in [linear, rbf], `C` in [1,10,50], `epsilon` in [0.01,0.05,0.1] | `kernel='rbf', C=10, epsilon=0.05` | RBF vuot han linear (du lieu phi tuyen - xem Hinh 7); C=10 can bang giua underfit (C nho) va overfit/cham (C lon) |
| Linear Regression | (baseline, khong tune) | mac dinh | Dung lam moc so sanh |


In [ ]:
def build_preprocessor():
    return ColumnTransformer(transformers=[
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
    ])

models = {
    'linear_regression': LinearRegression(),
    'decision_tree': DecisionTreeRegressor(max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE),
    'random_forest': RandomForestRegressor(n_estimators=300, max_depth=16, min_samples_leaf=2,
                                            n_jobs=-1, random_state=RANDOM_STATE),
    'svr': SVR(kernel='rbf', C=10, epsilon=0.05),
}
labels = {'linear_regression': 'Linear Regression (baseline)', 'decision_tree': 'Decision Tree',
          'random_forest': 'Random Forest', 'svr': 'SVR'}

In [ ]:
import time, joblib
results, fitted = [], {}
for key, base_model in models.items():
    pipe = Pipeline([('preprocess', build_preprocessor()), ('model', base_model)])
    ttr = TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)
    t0 = time.time(); ttr.fit(X_train, y_train); train_time = time.time() - t0
    t0 = time.time(); y_pred_test = ttr.predict(X_test); predict_time = (time.time()-t0)/len(X_test)
    y_pred_train = ttr.predict(X_train)
    mae_te, rmse_te, r2_te = (mean_absolute_error(y_test, y_pred_test),
                               mean_squared_error(y_test, y_pred_test)**0.5,
                               r2_score(y_test, y_pred_test))
    mae_tr, rmse_tr, r2_tr = (mean_absolute_error(y_train, y_pred_train),
                               mean_squared_error(y_train, y_pred_train)**0.5,
                               r2_score(y_train, y_pred_train))
    fitted[key] = ttr
    results.append(dict(model=labels[key], key=key, MAE_test=round(mae_te,2), RMSE_test=round(rmse_te,2),
                         R2_test=round(r2_te,4), MAE_train=round(mae_tr,2), RMSE_train=round(rmse_tr,2),
                         R2_train=round(r2_tr,4), train_time_s=round(train_time,2),
                         predict_time_ms=round(predict_time*1000,4)))
    print(f'[{labels[key]}] MAE={mae_te:.1f} RMSE={rmse_te:.1f} R2={r2_te:.4f} time={train_time:.1f}s')


[Linear Regression (baseline)] MAE=100.2 RMSE=344.7 R2=0.9676 time=0.6s
[Decision Tree] MAE=217.1 RMSE=611.6 R2=0.8980 time=0.2s
[Random Forest] MAE=136.1 RMSE=457.5 R2=0.9429 time=55.7s
[SVR] MAE=83.3 RMSE=289.0 R2=0.9772 time=35.9s


In [ ]:
metrics_df = pd.DataFrame(results).sort_values('R2_test', ascending=False)
metrics_df

model,key,MAE_test,RMSE_test,R2_test,MAE_train,RMSE_train,R2_train,train_time_s,predict_time_ms_per_row,model_size_MB
SVR,svr,83.3,289.05,0.9772,55.3,157.32,0.9919,35.89,0.3022,1.17
Linear Regression (baseline),linear_regression,100.18,344.67,0.9676,74.75,222.69,0.9837,0.62,0.0028,0.05
Random Forest,random_forest,136.12,457.53,0.9429,120.71,448.41,0.9339,55.67,0.0409,81.7
Decision Tree,decision_tree,217.15,611.64,0.898,202.26,572.68,0.8921,0.18,0.0027,0.13


## Xuat model tot nhat + schema + metadata

Model tot nhat theo R2 tren tap test se duoc luu thanh **mot file** `model.joblib` (bao gom ca pipeline tien xu ly, nho `TransformedTargetRegressor(regressor=Pipeline(...))`), kem `schema.json` va `metadata.json`.

**Cach export ra khoi Colab:** Sau khi chay xong o Colab, tai 3 file nay ve (`files.download(...)`) hoac luu vao Google Drive da mount, roi copy thu cong (hoac bang script dong bo / `git add` + `git push`) vao dung duong dan `ai-models/models/` trong repo - day la noi AI Service se doc luc khoi dong container.

In [ ]:
import json, platform, sklearn
best_key = metrics_df.iloc[0]['key']
best_model = fitted[best_key]
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/model.joblib', compress=3)
print('Model tot nhat:', labels[best_key])
print('Da luu ../models/model.joblib, kich thuoc (MB):',
      round(os.path.getsize('../models/model.joblib')/1024/1024, 2))
# schema.json va metadata.json: xem chi tiet cach sinh trong ai-models/src/train.py


Model tot nhat: SVR
Da luu ../models/model.joblib, kich thuoc (MB): 0.18


### Toan bo noi dung `train.py` (script tuong duong, dung khi chay ngoai Colab / trong Docker build)

In [ ]:
# -*- coding: utf-8 -*-
"""
train.py - Huan luyen 4 model hoi quy du doan gia xe cu.
Chay: python train.py
Dau ra:
  - ai-models/models/model.joblib     (pipeline tien xu ly + model tot nhat)
  - ai-models/models/schema.json
  - ai-models/models/metadata.json
  - docs/figures/08_model_comparison.png, 09_residuals.png
  - training metrics in ra man hinh + luu docs/metrics.csv
"""
import os
import sys
import json
import time
import platform
import numpy as np
import pandas as pd
import sklearn
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sys.path.insert(0, os.path.dirname(__file__))
from data_utils import (load_raw, clean_dataframe, FEATURE_COLUMNS, TARGET_COLUMN,
                         NUMERIC_FEATURES, CATEGORICAL_FEATURES)

HERE = os.path.dirname(__file__)
DATA_PATH = os.path.join(HERE, "..", "data", "data.csv")
MODELS_DIR = os.path.join(HERE, "..", "models")
FIG_DIR = os.path.join(HERE, "..", "..", "docs", "figures")
DOCS_DIR = os.path.join(HERE, "..", "..", "docs")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
RANDOM_STATE = 42


def build_preprocessor():
    return ColumnTransformer(transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])


def build_models():
    """4 model, gom 1 baseline. TransformedTargetRegressor: log1p(price) khi fit, expm1 khi predict."""
    models = {
        "linear_regression": LinearRegression(),
        "decision_tree": DecisionTreeRegressor(max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE),
        "random_forest": RandomForestRegressor(n_estimators=300, max_depth=16, min_samples_leaf=2,
                                                n_jobs=-1, random_state=RANDOM_STATE),
        "svr": SVR(kernel="rbf", C=10, epsilon=0.05),
    }
    return models


def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2


def main():
    raw = load_raw(DATA_PATH)
    df = clean_dataframe(raw)
    print("Du lieu sach:", df.shape)

    X = df[FEATURE_COLUMNS]
    y = df[TARGET_COLUMN].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE)
    print("Train:", X_train.shape, "Test:", X_test.shape)

    preprocessor = build_preprocessor()
    models = build_models()

    results = []
    fitted = {}
    labels = {
        "linear_regression": "Linear Regression (baseline)",
        "decision_tree": "Decision Tree",
        "random_forest": "Random Forest",
        "svr": "SVR",
    }

    for key, base_model in models.items():
        pipe = Pipeline(steps=[
            ("preprocess", build_preprocessor()),
            ("model", base_model),
        ])
        ttr = TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

        t0 = time.time()
        ttr.fit(X_train, y_train)
        train_time = time.time() - t0

        t0 = time.time()
        y_pred_test = ttr.predict(X_test)
        predict_time = (time.time() - t0) / max(len(X_test), 1)

        y_pred_train = ttr.predict(X_train)

        mae_test, rmse_test, r2_test = evaluate(y_test, y_pred_test)
        mae_train, rmse_train, r2_train = evaluate(y_train, y_pred_train)

        tmp_path = os.path.join(MODELS_DIR, f"_tmp_{key}.joblib")
        joblib.dump(ttr, tmp_path)
        size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
        os.remove(tmp_path)

        fitted[key] = ttr
        results.append({
            "model": labels[key],
            "key": key,
            "MAE_test": round(mae_test, 2),
            "RMSE_test": round(rmse_test, 2),
            "R2_test": round(r2_test, 4),
            "MAE_train": round(mae_train, 2),
            "RMSE_train": round(rmse_train, 2),
            "R2_train": round(r2_train, 4),
            "train_time_s": round(train_time, 2),
            "predict_time_ms_per_row": round(predict_time * 1000, 4),
            "model_size_MB": round(size_mb, 2),
        })
        print(f"[{labels[key]}] MAE={mae_test:.1f} RMSE={rmse_test:.1f} R2={r2_test:.4f} "
              f"(train R2={r2_train:.4f}) time={train_time:.1f}s size={size_mb:.1f}MB")

    metrics_df = pd.DataFrame(results).sort_values("R2_test", ascending=False)
    metrics_df.to_csv(os.path.join(DOCS_DIR, "metrics.csv"), index=False)
    print("\n=== Bang so sanh (sap xep theo R2 test) ===")
    print(metrics_df.to_string(index=False))

    best_key = metrics_df.iloc[0]["key"]
    best_model = fitted[best_key]
    print(f"\n>>> Model tot nhat: {labels[best_key]} ({best_key})")

    # ---- Hinh 8: so sanh model ----
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    order = metrics_df["model"]
    axes[0].bar(order, metrics_df["MAE_test"], color="#3a7ca5")
    axes[0].set_title("MAE tren tap test (trieu VND, thap hon = tot hon)")
    plt.setp(axes[0].get_xticklabels(), rotation=20, ha="right")
    axes[1].bar(order, metrics_df["R2_test"], color="#2f6690")
    axes[1].set_title("R2 tren tap test (cao hon = tot hon)")
    axes[1].set_ylim(0, 1)
    plt.setp(axes[1].get_xticklabels(), rotation=20, ha="right")
    fig.suptitle("Hinh 8: So sanh 4 model hoi quy")
    fig.savefig(os.path.join(FIG_DIR, "08_model_comparison.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)

    # ---- Hinh 9: bieu do phan du cua model tot nhat ----
    y_pred_best = best_model.predict(X_test)
    residuals = y_test - y_pred_best
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].scatter(y_pred_best, residuals, s=8, alpha=0.3, color="#2f6690")
    axes[0].axhline(0, color="red", linewidth=1)
    axes[0].set_xlabel("Gia du doan (trieu VND)")
    axes[0].set_ylabel("Phan du (thuc te - du doan)")
    axes[0].set_title(f"Phan du - {labels[best_key]}")
    axes[1].hist(residuals, bins=60, color="#3a7ca5")
    axes[1].set_title("Phan bo phan du")
    fig.suptitle("Hinh 9: Phan tich loi cua model tot nhat")
    fig.savefig(os.path.join(FIG_DIR, "09_residuals.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)

    # ---- Xuat model + schema + metadata ----
    model_path = os.path.join(MODELS_DIR, "model.joblib")
    joblib.dump(best_model, model_path, compress=3)
    print("Da luu model:", model_path, f"({os.path.getsize(model_path)/1024/1024:.2f} MB)")

    schema = {
        "target": TARGET_COLUMN,
        "target_unit": "million_vnd",
        "features": [
            {"name": "brand", "dtype": "category", "example": "Toyota"},
            {"name": "model", "dtype": "category", "example": "Camry 2.5Q"},
            {"name": "series", "dtype": "category", "allowed_values": sorted(df["series"].unique().tolist())},
            {"name": "year", "dtype": "int", "min": int(df["year"].min()), "max": int(df["year"].max())},
            {"name": "driven_kms", "dtype": "float", "min": float(df["driven_kms"].min()), "max": float(df["driven_kms"].max())},
            {"name": "assemble_place", "dtype": "category", "allowed_values": sorted(df["assemble_place"].unique().tolist())},
            {"name": "engine_type", "dtype": "category", "allowed_values": sorted(df["engine_type"].unique().tolist())},
            {"name": "transmission", "dtype": "category", "allowed_values": sorted(df["transmission"].unique().tolist())},
            {"name": "num_of_door", "dtype": "int", "min": int(df["num_of_door"].min()), "max": int(df["num_of_door"].max())},
            {"name": "num_of_seat", "dtype": "int", "min": int(df["num_of_seat"].min()), "max": int(df["num_of_seat"].max())},
        ],
    }
    with open(os.path.join(MODELS_DIR, "schema.json"), "w", encoding="utf-8") as f:
        json.dump(schema, f, ensure_ascii=False, indent=2)

    best_row = metrics_df.iloc[0].to_dict()
    metadata = {
        "model_name": labels[best_key],
        "model_key": best_key,
        "model_version": "1.0.0",
        "trained_at": pd.Timestamp.now().isoformat(),
        "metrics_test": {"MAE": best_row["MAE_test"], "RMSE": best_row["RMSE_test"], "R2": best_row["R2_test"]},
        "metrics_train": {"MAE": best_row["MAE_train"], "RMSE": best_row["RMSE_train"], "R2": best_row["R2_train"]},
        "all_models_compared": results,
        "target_unit": "million_vnd",
        "n_rows_train": int(len(X_train)),
        "n_rows_test": int(len(X_test)),
        "random_state": RANDOM_STATE,
        "library_versions": {
            "python": platform.python_version(),
            "scikit-learn": sklearn.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "joblib": joblib.__version__,
        },
    }
    with open(os.path.join(MODELS_DIR, "metadata.json"), "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print("Da luu schema.json va metadata.json trong", MODELS_DIR)


if __name__ == "__main__":
    main()
